# 3장 1강: CNN의 구조와 원리 — 실습문제 풀이

> **풀이 기준**: 업로드된 `jena_climate_2009_2016.csv`(420,551행 × 15열)의 처음 16행으로 코드를 실제 실행한 결과입니다. 출력은 소수 넷째 자리에서 반올림했습니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 문제의 필터·stride·padding 설정을 그대로 사용했습니다.

---

## 실습 준비 (제공 코드)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("jena_climate_2009_2016.csv", nrows=16)
times = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
x = df["T (degC)"].iloc[:8].to_numpy(dtype=float)
x_home = df["T (degC)"].iloc[8:16].to_numpy(dtype=float)

print("읽은 행과 열:", df.shape)
print(df[["Date Time", "T (degC)"]].to_string(index=False))
print("선택 구간 간격(분):", times.diff().dropna().dt.total_seconds().unique() / 60)
print("선택 기온의 결측값 수:", df["T (degC)"].isna().sum())
print("실습 입력:", x)
print("과제 입력:", x_home)

읽은 행과 열: (16, 15)
          Date Time  T (degC)
01.01.2009 00:10:00     -8.02
01.01.2009 00:20:00     -8.41
01.01.2009 00:30:00     -8.51
01.01.2009 00:40:00     -8.31
01.01.2009 00:50:00     -8.27
01.01.2009 01:00:00     -8.05
01.01.2009 01:10:00     -7.62
01.01.2009 01:20:00     -7.62
01.01.2009 01:30:00     -7.91
01.01.2009 01:40:00     -8.43
01.01.2009 01:50:00     -8.76
01.01.2009 02:00:00     -8.88
01.01.2009 02:10:00     -8.85
01.01.2009 02:20:00     -8.83
01.01.2009 02:30:00     -8.66
01.01.2009 02:40:00     -8.66
선택 구간 간격(분): [10.]
선택 기온의 결측값 수: 0
실습 입력: [-8.02 -8.41 -8.51 -8.31 -8.27 -8.05 -7.62 -7.62]
과제 입력: [-7.91 -8.43 -8.76 -8.88 -8.85 -8.83 -8.66 -8.66]


In [2]:
def conv1d_valid(values, kernel):
    results = []
    for i in range(len(values) - len(kernel) + 1):
        window = values[i:i + len(kernel)]
        value = np.sum(window * kernel)
        results.append(value)
    return np.array(results)


def pool_pairs(values):
    maximums = []
    averages = []
    for i in range(0, len(values), 2):
        window = values[i:i + 2]
        maximums.append(np.max(window))
        averages.append(np.mean(window))
    return np.array(maximums), np.array(averages)


print("합성곱 함수와 풀링 함수 준비 완료")

합성곱 함수와 풀링 함수 준비 완료


---

## 실습 1. Convolution과 특징 추출

### 문제 1-1. 같은 기온에 두 가지 필터 적용하기

In [3]:
k_mean = np.array([1/3, 1/3, 1/3])
k_diff = np.array([-1, 0, 1])

f_mean = conv1d_valid(x, k_mean)
f_diff = conv1d_valid(x, k_diff)

print("평균 필터 특징 맵:", np.round(f_mean, 4), " 길이:", len(f_mean))
print("차이 필터 특징 맵:", np.round(f_diff, 4), " 길이:", len(f_diff))

평균 필터 특징 맵: [-8.3133 -8.41   -8.3633 -8.21   -7.98   -7.7633]  길이: 6
차이 필터 특징 맵: [-0.49  0.1   0.24  0.26  0.65  0.43]  길이: 6


#### 답변

**수식 및 계산 과정 (첫 구간 x[:3] = [−8.02, −8.41, −8.51])**

- 평균 필터: (−8.02)(1/3) + (−8.41)(1/3) + (−8.51)(1/3) = (−24.94) / 3 = **−8.3133**
- 차이 필터: (−8.02)(−1) + (−8.41)(0) + (−8.51)(1) = 8.02 + 0 − 8.51 = **−0.49**

차이 특징 맵의 부호 해석:

| 구간 | 시작 → 끝 기온 | 차이 | 해석 |
|---|---|---:|---|
| 1 | −8.02 → −8.51 | −0.49 | 끝이 시작보다 낮음 |
| 2 | −8.41 → −8.31 | +0.10 | 끝이 시작보다 높음 |
| 3 | −8.51 → −8.27 | +0.24 | 끝이 시작보다 높음 |
| 4 | −8.31 → −8.05 | +0.26 | 끝이 시작보다 높음 |
| 5 | −8.27 → −7.62 | +0.65 | 끝이 시작보다 높음 |
| 6 | −8.05 → −7.62 | +0.43 | 끝이 시작보다 높음 |

**Q1. 입력 기온, 필터, 특징 맵은 각각 무엇인가요? 길이 8인 입력의 특징 맵 길이가 6인 이유는 무엇인가요?**

- 입력 기온은 10분 간격으로 관측한 기온 8개입니다. 필터는 어떤 패턴에 반응할지 정하는 가중치 3개입니다. 특징 맵은 필터를 한 칸씩 옮기며 계산한 결과를 모은 배열입니다.
- 길이 3짜리 필터가 입력 안에 완전히 들어가는 시작 위치는 0~5의 6곳뿐이므로 8 − 3 + 1 = **6**입니다(stride 1, padding 없음).

**Q2. 두 필터의 첫 번째 출력값은 각각 얼마인가요?**

- 평균 필터 **−8.3133**, 차이 필터 **−0.49**입니다(위 계산).

**Q3. 차이 특징 맵에서 음수와 양수는 무엇을 뜻하나요? 양수라면 세 관측값이 모두 연속해서 증가했다는 뜻인가요?**

- 음수는 구간 끝(20분 뒤)의 기온이 시작보다 **낮다**는 뜻이고, 양수는 **높다**는 뜻입니다.
- 연속 증가를 뜻하지는 않습니다. 가운데 값의 가중치가 0이라 가운데 기온은 결과에 반영되지 않습니다. 예를 들어 구간 3(−8.51, −8.31, −8.27)은 연속 증가했지만, 구간 2(−8.41, −8.51, −8.31)는 내려갔다가 올라갔는데도 +0.10입니다.

**Q4. 같은 입력에 다른 필터를 적용하면 왜 결과가 달라지나요? 이번 필터와 실제 CNN에서 학습하는 필터의 차이도 설명하세요.**

- 필터 값이 곧 각 위치에 곱하는 가중치이므로, 가중치가 다르면 곱하고 더한 결과가 달라집니다. 평균 필터는 주변 기온의 **수준**을, 차이 필터는 **변화 방향**을 뽑아냅니다.
- 이번 필터는 사람이 정해 고정한 값입니다. 실제 CNN에서는 필터 가중치(와 편향)가 **데이터로부터 학습**되어, 문제를 푸는 데 유용한 패턴을 모델이 스스로 찾습니다.

---

## 실습 2. Pooling과 CNN 이미지 분류 구조

### 문제 2-1. 특징 맵을 ReLU와 Pooling으로 요약하기

In [4]:
activated = np.maximum(0.0, f_diff)
max_map, avg_map = pool_pairs(activated)

print("합성곱 결과 :", np.round(f_diff, 4), " 길이:", len(f_diff))
print("ReLU 결과   :", np.round(activated, 4), " 길이:", len(activated))
print("Max Pooling :", np.round(max_map, 4), " 길이:", len(max_map))
print("Avg Pooling :", np.round(avg_map, 4), " 길이:", len(avg_map))

합성곱 결과 : [-0.49  0.1   0.24  0.26  0.65  0.43]  길이: 6
ReLU 결과   : [0.   0.1  0.24 0.26 0.65 0.43]  길이: 6
Max Pooling : [0.1  0.26 0.65]  길이: 3
Avg Pooling : [0.05 0.25 0.54]  길이: 3


#### 답변

**수식 및 계산 과정**

ReLU: [−0.49, 0.10, 0.24, 0.26, 0.65, 0.43] → **[0, 0.10, 0.24, 0.26, 0.65, 0.43]**

| 묶음 | 값 | Max | Average |
|---|---|---:|---:|
| 1 | [0, 0.10] | 0.10 | (0 + 0.10)/2 = 0.05 |
| 2 | [0.24, 0.26] | 0.26 | (0.24 + 0.26)/2 = 0.25 |
| 3 | [0.65, 0.43] | 0.65 | (0.65 + 0.43)/2 = 0.54 |

**Q1. ReLU는 이번 차이 특징 맵에서 어떤 정보를 남기고 어떤 정보를 없애나요?**

- 기온이 **오른 구간(양수)의 크기**는 그대로 남기고, 기온이 내린 구간(음수 −0.49)은 0으로 바꿔 없앱니다. 즉 상승 반응만 남깁니다.

**Q2. Max Pooling과 Average Pooling의 세 결과는 각각 얼마인가요? 길이는 왜 6에서 3이 되나요?**

- Max: **[0.10, 0.26, 0.65]**, Average: **[0.05, 0.25, 0.54]**입니다.
- 두 개씩 겹치지 않게(크기 2, stride 2) 묶어 대표값 하나만 남기므로 6 / 2 = **3**이 됩니다.

**Q3. 합성곱 필터와 이 두 풀링의 역할은 어떻게 다른가요? 일반적인 Max/Average Pooling에 학습할 가중치가 있나요?**

- 합성곱 필터는 가중치를 곱해 **특정 패턴을 찾아내는** 역할이고, 풀링은 이미 만든 특징 맵을 **작은 구역별 대표값으로 요약해 크기를 줄이는** 역할입니다.
- 일반적인 Max/Average Pooling은 최댓값 또는 평균을 구하는 고정된 규칙일 뿐, **학습할 가중치가 없습니다**.

**Q4. 풀링 결과만 보고 입력 특징 맵을 정확히 복원할 수 있나요? 값의 개수를 줄이면 항상 분류가 더 잘되나요?**

- 복원할 수 없습니다. 예를 들어 Max 0.65만으로는 짝이 0.43이었는지 알 수 없고, 평균 0.25도 [0.24, 0.26]과 [0, 0.5]에서 똑같이 나옵니다.
- 항상 좋아지지는 않습니다. 계산량을 줄이고 작은 위치 변화에 덜 민감해지는 장점이 있지만, 세부 정보를 잃으므로 지나치면 분류에 필요한 정보까지 사라질 수 있습니다.

---

### 문제 2-2. CNN 기반 이미지 분류의 처리 흐름 연결하기 (코드 없음)

#### 답변

**단계별 순서·크기·역할 표**

| 순서 | 단계 | 출력 크기 (높이×너비×채널) | 하는 일 |
|---:|---|---|---|
| 1 | 입력 이미지 | 8×8×1 | 흑백 이미지의 픽셀 값을 받습니다. |
| 2 | Convolution | 6×6×2 | 3×3 필터 2개를 이동하며 곱하고 더해 특징 맵 2개를 만듭니다 (8−3+1 = 6). |
| 3 | ReLU | 6×6×2 | 음수를 0으로 바꿔 양의 반응만 남깁니다. 크기는 그대로입니다. |
| 4 | Pooling | 3×3×2 | 2×2 구역마다 최댓값만 남겨 각 특징 맵을 요약합니다 (6 / 2 = 3). |
| 5 | Flatten | 18 | 3×3×2 = 18개 값을 한 줄로 펼칩니다. 값의 개수는 그대로입니다. |
| 6 | Dense | 3 | 18개 특징을 가중합해 클래스 A·B·C의 점수 3개를 만듭니다. |

**Q1. 올바른 처리 순서와 각 단계의 출력 크기는 무엇인가요?**

- 입력 이미지(8×8×1) → Convolution(6×6×2) → ReLU(6×6×2) → Pooling(3×3×2) → Flatten(18) → Dense(3)입니다.

**Q2. 합성곱·ReLU·풀링·Flatten·Dense는 각각 어떤 역할을 하나요?**

- 합성곱: 주변 픽셀의 패턴(모서리, 선 등)을 필터로 찾아냅니다.
- ReLU: 비선형성을 더하고 음의 반응을 0으로 만듭니다.
- 풀링: 특징 맵을 요약해 크기를 줄입니다.
- Flatten: 다차원 특징을 Dense가 받을 수 있게 1차원으로 펼칩니다.
- Dense: 펼친 특징을 모두 연결해 클래스별 점수를 계산합니다.

**Q3. 예시 점수에서 선택할 클래스는 무엇인가요? B의 점수 1.4를 140%의 확률로 해석해도 되나요?**

- 가장 높은 점수(1.4)를 가진 **클래스 B**를 선택합니다.
- 안 됩니다. 이 점수는 확률로 변환하지 않은 값이라 1을 넘거나 음수(C: −0.3)일 수도 있습니다. 확률로 해석하려면 Softmax 등으로 변환해야 합니다.

**Q4. Jena 기온 실습과 이미지 분류 구조는 무엇이 같고 무엇이 다른가요? 기온 특징 맵을 계산한 것만으로 이미지 분류 모델 학습까지 완료했다고 할 수 있나요?**

- 같은 점: 필터를 이동하며 곱하고 더해 특징 맵을 만들고, ReLU와 풀링으로 요약하는 원리가 같습니다.
- 다른 점: 기온은 시간 방향 1차원 입력에 1차원 필터를 쓰고, 이미지는 높이·너비 2차원 입력에 2차원 필터를 씁니다. 또 기온 실습에는 Flatten·Dense와 클래스 출력이 없었습니다.
- 학습을 완료했다고 할 수 없습니다. 기온 실습은 고정 필터로 특징을 계산했을 뿐이고, 이미지 분류는 구조만 설명했으며 데이터로 필터·가중치를 학습하지 않았습니다.

---

## 기초 과제. 다른 기온 구간에 같은 계산 적용하기

### 문제 3-1. 합성곱부터 Max Pooling까지 이어서 계산하기

In [5]:
home_filter = np.array([-1, 0, 1])
home_feature = conv1d_valid(x_home, home_filter)
home_relu = np.maximum(0.0, home_feature)
home_max, home_avg_unused = pool_pairs(home_relu)

print("입력        :", x_home, " 길이:", len(x_home))
print("합성곱 결과 :", np.round(home_feature, 4), " 길이:", len(home_feature))
print("ReLU 결과   :", np.round(home_relu, 4), " 길이:", len(home_relu))
print("Max Pooling :", np.round(home_max, 4), " 길이:", len(home_max))

입력        : [-7.91 -8.43 -8.76 -8.88 -8.85 -8.83 -8.66 -8.66]  길이: 8
합성곱 결과 : [-0.85 -0.45 -0.09  0.05  0.19  0.17]  길이: 6
ReLU 결과   : [0.   0.   0.   0.05 0.19 0.17]  길이: 6
Max Pooling : [0.   0.05 0.19]  길이: 3


#### 답변

**수식 및 계산 과정**

- 첫 합성곱: (−7.91)(−1) + (−8.43)(0) + (−8.76)(1) = 7.91 − 8.76 = **−0.85**
- ReLU: [−0.85, −0.45, −0.09, 0.05, 0.19, 0.17] → [0, 0, 0, 0.05, 0.19, 0.17]
- Max Pooling: max(0, 0) = **0**, max(0, 0.05) = **0.05**, max(0.19, 0.17) = **0.19**

**Q1. 첫 구간의 합성곱 결과는 얼마이며, 어떻게 계산하나요?**

- **−0.85**입니다. 첫 구간 [−7.91, −8.43, −8.76]에 [−1, 0, 1]을 같은 위치끼리 곱해 더하면 7.91 + 0 − 8.76 = −0.85입니다. 20분 동안 기온이 0.85℃ 내려갔다는 뜻입니다.

**Q2. ReLU를 적용한 특징 맵과 Max Pooling 결과는 각각 무엇인가요? 0인 풀링 결과는 기온이 0℃라는 뜻인가요?**

- ReLU: **[0, 0, 0, 0.05, 0.19, 0.17]**, Max Pooling: **[0, 0.05, 0.19]**입니다.
- 아닙니다. 0은 기온이 아니라 "그 구역에 **기온 상승 반응이 없었다**(두 구간 모두 하강)"는 뜻입니다. 실제 기온은 −8℃대입니다.

**Q3. 네 단계의 길이는 어떻게 바뀌나요? 앞선 실습과 결과 숫자가 다른 이유는 무엇인가요?**

- 입력 8 → 합성곱 6 → ReLU 6 → Max Pooling 3입니다.
- 필터·ReLU·풀링 규칙은 같고 **입력 기온 구간이 달라졌기** 때문입니다. 실습 구간은 대체로 기온이 오르는 흐름이었지만, 과제 구간은 앞부분에서 내려가다가 뒷부분에서 조금 오르는 흐름입니다.

**Q4. 이미지 분류 구조에서는 이렇게 추출·요약한 특징을 어떤 단계로 연결해 클래스별 점수를 만드나요?**

- 풀링 결과를 **Flatten**으로 한 줄로 펼친 뒤, **Dense(완전연결층)**에서 가중합해 클래스별 점수를 만듭니다.

---

## 실습 마무리

**1. Convolution, Filter, Feature Map은 각각 무엇인가요?**

- Convolution: 입력의 작은 구간과 필터를 같은 위치끼리 곱한 뒤 더하는 계산입니다. 예: [−8.02, −8.41, −8.51]과 [−1, 0, 1] → −0.49
- Filter: 어떤 패턴에 반응할지 정하는 가중치 묶음입니다. 예: 차이 필터 [−1, 0, 1]
- Feature Map: 필터를 모든 위치에 적용한 결과 배열입니다. 예: [−0.49, 0.10, 0.24, 0.26, 0.65, 0.43]

**2. 입력 길이 8, 필터 길이 3, stride 1, padding 없음일 때 출력 길이는 왜 6인가요? 두 필터를 적용하면 무엇이 두 개 생기나요?**

- 필터가 입력 안에 완전히 들어가는 시작 위치가 8 − 3 + 1 = 6곳이기 때문입니다.
- 필터마다 특징 맵이 하나씩 생기므로 **특징 맵(채널)이 두 개** 생깁니다(f_mean, f_diff).

**3. 차이 필터 [−1, 0, 1]의 결과가 양수라는 것은 무엇을 뜻하나요? ReLU에서 0이 되었다면 원래 차이를 정확히 알 수 있나요?**

- 구간 끝 기온이 시작보다 높다(20분 동안 상승)는 뜻입니다.
- 알 수 없습니다. ReLU는 모든 음수와 0을 0으로 만들므로, 원래 값이 −0.85였는지 −0.09였는지 구별할 수 없습니다.

**4. [0.24, 0.26]에 Max Pooling과 Average Pooling을 적용한 결과를 각각 계산하세요. 풀링이 남기는 정보와 잃는 정보도 설명하세요.**

- Max = **0.26**, Average = (0.24 + 0.26)/2 = **0.25**입니다.
- 남기는 정보: 구역에서 가장 강한 반응(Max) 또는 평균 수준(Average). 잃는 정보: 구역 안의 개별 값과 정확한 위치입니다.

**5. 이미지 분류 예시의 처리 순서와 출력 크기를 적으세요. Flatten과 Dense는 어떻게 다른가요?**

- 입력 8×8×1 → Convolution 6×6×2 → ReLU 6×6×2 → Pooling 3×3×2 → Flatten 18 → Dense 3
- Flatten은 값을 한 줄로 **재배열만** 하며 학습할 가중치가 없고 값의 개수도 그대로입니다. Dense는 가중치·편향으로 **새로운 값(클래스 점수)을 계산**하는 학습 층입니다.

**6. 이번 실습에서 이미지 분류 모델을 학습했나요? 공통점·차이점, 고정 필터와 학습되는 필터의 차이를 설명하세요.**

- 학습하지 않았습니다.
- 공통점: 필터 이동 → 곱하고 더하기 → ReLU → 풀링이라는 특징 추출 원리입니다.
- 차이점: 기온은 1차원 시계열, 이미지는 2차원(높이·너비)입니다.
- 이번 필터는 사람이 정한 고정값이고, 실제 CNN의 필터는 데이터를 이용해 역전파로 조정되는 학습 파라미터입니다.